
 Notebook: silver_products
 Layer: Silver (Transformation)
 Source: bronze.products

 Description:
 - Cleans and standardizes product data
 - Handles null values and data types
 - Deduplicates records (1 row per product_id)
 - Prepares dataset for downstream joins

 Output:
 - Table: silver.products


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

 Step 1: Read data from Bronze layer

In [0]:
df = spark.table("bronze.products")

 Step 2: Data validation

In [0]:
# Check schema and sample data
display(df)

# Count total records
df.count()

 Step 3: Basic cleaning

In [0]:
# Remove records with null product_id (invalid primary key)
df_clean = df.filter(F.col("product_id").isNotNull())

 Step 4: Standardization

In [0]:
# Ensure consistent data types and column naming

df_clean = df_clean.withColumn("price", F.col("price").cast("double"))

 Step 5: Deduplication

In [0]:
# Keep only one record per product_id
# If duplicates exist, keep the latest based on ingestion_timestamp

window_spec = Window.partitionBy("product_id").orderBy(F.col("ingestion_timestamp").desc())

df_dedup = df_clean.withColumn("row_num", F.row_number().over(window_spec)) \
                  .filter(F.col("row_num") == 1) \
                  .drop("row_num")

 Step 6: Final validation

In [0]:
display(df_dedup)

df_dedup.count()

 Step 7: Write to Silver table

In [0]:
df_dedup.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.products")

 Step 8: Verify Silver table

In [0]:
display(spark.table("silver.products"))

In [0]:
spark.table("silver.products").select("product_id").show(5)